In [2]:
import tensorflow as tf
print(tf.__version__, tf.config.list_physical_devices())

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# ============================================================
# EDIT THIS ONE LINE to match where the project lives in YOUR Google Drive.
# ============================================================
PROJECT_PATH = "/content/drive/MyDrive/crop-disease-detection"

# Cell 1: mount Google Drive (a one-time authorisation popup will appear)
try:
    from google.colab import drive
except ImportError as err:
    raise RuntimeError(
        "This notebook must be run in Google Colab (google.colab is not available here). "
        "For local work, follow ml/README.md instead."
    ) from err

drive.mount('/content/drive')

RuntimeError: This notebook must be run in Google Colab (google.colab is not available here). For local work, follow ml/README.md instead.

In [ ]:
# Cell 2: move into the project folder and make sure the files we need are really there
import os

if not os.path.isdir(PROJECT_PATH):
    raise FileNotFoundError(
        f"PROJECT_PATH not found: {PROJECT_PATH}\n"
        "Fix the PROJECT_PATH line in the first cell so it points at your crop-disease-detection folder on Drive."
    )

REQUIREMENTS_FILE = os.path.join(PROJECT_PATH, "ml", "requirements.txt")
if not os.path.isfile(REQUIREMENTS_FILE):
    raise FileNotFoundError(
        f"Missing {REQUIREMENTS_FILE}\n"
        "Make sure the whole project (including the ml/ folder) is uploaded to Drive at PROJECT_PATH."
    )

os.chdir(PROJECT_PATH)
print("Working directory:", os.getcwd())

In [ ]:
# Cell 3: install the project's ML dependencies (equivalent to: !pip install -r {PROJECT_PATH}/ml/requirements.txt -q)
import subprocess, sys
from importlib import metadata

if not ((3, 10) <= sys.version_info[:2] <= (3, 11)):
    print(f"NOTE: Python {sys.version_info.major}.{sys.version_info.minor} detected; the project targets 3.10/3.11 "
          "(see ml/README.md). If the install below fails, this is the likely reason.")

# numpy version already loaded in this session (Colab preloads it), to detect a needed restart
loaded_numpy = sys.modules["numpy"].__version__ if "numpy" in sys.modules else None

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", REQUIREMENTS_FILE, "-q"],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print(result.stdout[-3000:])
    print(result.stderr[-3000:])
    raise RuntimeError("pip install failed - see the output above. Fix the problem and re-run this cell.")

installed_numpy = metadata.version("numpy")
if loaded_numpy is not None and loaded_numpy != installed_numpy:
    print(f"Install finished, but numpy changed ({loaded_numpy} -> {installed_numpy}) after it was already loaded.\n"
          "ACTION REQUIRED: Runtime -> Restart session, then re-run Cell 1 and Cell 2, SKIP this cell, and run Cell 4.")
else:
    print("Install finished successfully.")

In [ ]:
# Cell 4: verify that a GPU is attached
import subprocess
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print("TensorFlow version:", tf.__version__)
print("GPU available:", gpus)

if not gpus:
    print("\n" + "=" * 70)
    print("WARNING: NO GPU DETECTED - training will be extremely slow on CPU.")
    try:
        has_nvidia = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
    except FileNotFoundError:
        has_nvidia = False
    if has_nvidia:
        print("A GPU is attached to this runtime, but TensorFlow cannot use it")
        print("(usually a CUDA/cuDNN mismatch with the pinned TensorFlow version).")
        print("Ask the team to check the TensorFlow/CUDA setup in ml/requirements.txt.")
    else:
        print("Enable one via: Runtime -> Change runtime type -> Hardware accelerator -> GPU,")
        print("then Runtime -> Restart session and re-run this notebook from the top.")
    print("=" * 70)
else:
    print("GPU check passed - ready for training.")

## Usage

Run this notebook **first** in every new Colab session. Every other notebook in `ml/notebooks/` should begin by running it, or at the very least by re-running its **Drive-mount** and **path-setup** cells (the first two code cells), so that the working directory and `PROJECT_PATH` are set correctly.

- Only the `PROJECT_PATH` line in the first cell ever needs editing.
- Colab sessions reset after inactivity, so repeat this setup after any disconnect.
- If Cell 3 asks for a restart (numpy/TensorFlow changed), use **Runtime -> Restart session**, re-run Cells 1-2, skip Cell 3, then run Cell 4.